# JobCompass — Notebook 02: Structured Profile & Job Extraction

## What this does

**Candidate side:**
- Reads `enriched_resumes.jsonl` from NB01 (vLLM OCR output)
- Calls Qwen2.5-7B-Instruct to extract structured profile: name, skills, experience, education, seniority
- Normalises skills via taxonomy (if available)
- Writes `candidates.jsonl`

**Job side:**
- Reads `job_descriptions.csv`
- Cleans HTML/EEO boilerplate, parses all CSV columns
- Calls same LLM to extract structured fields + infer domain tags from content
- Writes `jobs.jsonl`

## Key design decisions
- **Domain tags come from LLM only** — no folder/category shortcuts
- `source_category` stored for offline evaluation only, never used for inference
- Pure text LLM (Qwen2.5-7B-Instruct) — no vision overhead
- `embedding_text` pre-computed and stored; `description_vector` filled in NB03

## Server to start in WSL2 before running
```bash
python3 -m vllm.entrypoints.openai.api_server \
  --model /mnt/d/models/qwen-3b \
  --trust-remote-code \
  --dtype float16 \
  --max-model-len 8192 \
  --max-num-seqs 16 \
  --gpu-memory-utilization 0.85 \
  --port 8001 \
  --host 0.0.0.0
```
Download model first:
```bash
huggingface-cli download Qwen/Qwen2.5-7B-Instruct --local-dir /mnt/d/models/Qwen2.5-7B-Instruct --local-dir-use-symlinks False
```

In [1]:
# Cell 0 — Install dependencies
import subprocess, sys

for pkg in ["openai", "aiohttp", "orjson", "tqdm", "pandas", "nest_asyncio", "requests"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    print(f"  {'OK' if r.returncode == 0 else 'FAIL'} {pkg}")
print("Done.")

  OK openai
  OK aiohttp
  OK orjson
  OK tqdm
  OK pandas
  OK nest_asyncio
  OK requests
Done.


In [2]:
# Cell 1 — Configuration
import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Paths
ENRICHED_JSONL      = "../data/extracted_text/enriched_resumes.jsonl"
JOBS_CSV            = "../data/raw/job_descriptions.csv"
PROCESSED_DIR       = "../data/processed"
CANDIDATES_JSONL    = f"{PROCESSED_DIR}/candidates.jsonl"
SKIPPED_CANDS_JSONL = f"{PROCESSED_DIR}/skipped_candidates.jsonl"
JOBS_JSONL          = f"{PROCESSED_DIR}/jobs.jsonl"
SKIPPED_JOBS_JSONL  = f"{PROCESSED_DIR}/skipped_jobs.jsonl"
TAXONOMY_PATH       = f"{PROCESSED_DIR}/skills_taxonomy.json"
PROGRESS_FILE       = f"{PROCESSED_DIR}/nb02_progress.json"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# vLLM — Qwen2.5-7B-Instruct on port 8001 (separate from NB01 OCR server on 8000)
VLLM_BASE_URL = "http://localhost:8001/v1"
VLLM_MODEL    = "/mnt/d/models/qwen-3b"

# Throughput — text-only calls are fast; increase if GPU util < 70%
CONCURRENT_REQUESTS  = 3
CHECKPOINT_EVERY     = 100
REQUEST_TIMEOUT_S    = 60
MAX_RETRIES          = 2
RETRY_BACKOFF_S      = 1.0

# Token budgets (8192 context: prompt ~1500t + output ~800t = safe headroom)
CANDIDATE_MAX_TOKENS = 900
JOB_MAX_TOKENS       = 700
RESUME_TEXT_TRUNCATE = 3500   # chars
JOB_TEXT_TRUNCATE    = 2500   # chars

# Quality gates
MIN_RESUME_CHARS   = 150
MIN_JOB_DESC_CHARS = 80
MIN_SKILLS_COUNT   = 1

# ── Job CSV limit ──────────────────────────────────────────────────────────────
# Set to None to process all rows. Set to an integer to cap at that many rows
# (applied after dedup/empty-drop, so final count may be slightly less).
MAX_JOBS = 5000

print("Config ready.")
print(f"  vLLM  : {VLLM_BASE_URL}  model={VLLM_MODEL}")
print(f"  Concurrent={CONCURRENT_REQUESTS}  CandTokens={CANDIDATE_MAX_TOKENS}  JobTokens={JOB_MAX_TOKENS}")
print(f"  Max jobs to process : {MAX_JOBS}")

Config ready.
  vLLM  : http://localhost:8001/v1  model=/mnt/d/models/qwen-3b
  Concurrent=3  CandTokens=900  JobTokens=700
  Max jobs to process : 5000


In [3]:
# Cell 2 — Verify vLLM server + input files
import requests

health_url = VLLM_BASE_URL.replace("/v1", "") + "/health"
try:
    r = requests.get(health_url, timeout=5)
    print(f"vLLM server : OK (HTTP {r.status_code})")
except Exception as e:
    raise RuntimeError(f"vLLM server unreachable at {health_url} — start it in WSL2 first.\n{e}")

model_id = requests.get(f"{VLLM_BASE_URL}/models", timeout=5).json()["data"][0]["id"]
print(f"Loaded model : {model_id}")

for path in [ENRICHED_JSONL, JOBS_CSV]:
    if not Path(path).exists():
        raise FileNotFoundError(f"Required input not found: {path}")
    print(f"  Found : {path}")

vLLM server : OK (HTTP 200)
Loaded model : /mnt/d/models/qwen-3b
  Found : ../data/extracted_text/enriched_resumes.jsonl
  Found : ../data/raw/job_descriptions.csv


In [4]:
# Cell 3 — Shared utilities: checkpoint, taxonomy, JSON parser, helpers
import orjson, re, json, time, asyncio, aiohttp, hashlib
from datetime import datetime, timezone
from pathlib import Path

# Checkpoint
if Path(PROGRESS_FILE).exists():
    with open(PROGRESS_FILE, "rb") as f:
        progress = orjson.loads(f.read())
    print(f"Checkpoint: candidates_done={len(progress.get('candidates_done',[]))}  jobs_done={len(progress.get('jobs_done',[]))}")
else:
    progress = {"candidates_done": [], "jobs_done": []}
    print("Starting fresh.")

def save_progress():
    with open(PROGRESS_FILE, "wb") as f:
        f.write(orjson.dumps(progress))

# Skills taxonomy
if Path(TAXONOMY_PATH).exists():
    with open(TAXONOMY_PATH, encoding="utf-8") as f:
        taxonomy_data = json.load(f)
    alias_map = {alias.lower().strip(): canonical
                 for canonical, aliases in taxonomy_data.items()
                 for alias in aliases}
    print(f"Taxonomy: {len(alias_map)} aliases -> {len(taxonomy_data)} canonical skills")
else:
    alias_map = {}
    print("Taxonomy: not found (skills stored as-is)")

def normalize_skill(raw: str) -> str:
    if not raw: return ""
    c = re.sub(r"\s+", " ", raw.strip().strip(".,;:()/[]{}")).strip()
    return "" if len(c) < 2 else alias_map.get(c.lower(), c)

def normalize_skills(lst: list) -> list:
    seen, out = set(), []
    for s in (lst or []):
        n = normalize_skill(str(s))
        if n and n.lower() not in seen:
            seen.add(n.lower()); out.append(n)
    return out

def parse_llm_json(raw: str) -> dict | None:
    """Robustly parse JSON from LLM — handles fences, surrounding text, truncation."""
    if not raw or not raw.strip(): return None
    text = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    text = re.sub(r"\s*```$", "", text).strip()
    for attempt in [text, (re.search(r"\{.*\}", text, re.DOTALL) or type('', (), {'group': lambda s, x: None})()).group(0)]:
        if not attempt: continue
        try: return json.loads(attempt)
        except Exception: pass
    # Fix truncated JSON
    try:
        t = text
        t += "]" * max(0, t.count("[") - t.count("]"))
        t += "}" * max(0, t.count("{") - t.count("}"))
        return json.loads(t)
    except Exception:
        return None

def safe_str(val, default: str = "") -> str:
    if val is None: return default
    s = str(val).strip()
    return default if s.lower() in ("nan", "none", "") else s

print("Utilities ready.")

Checkpoint: candidates_done=2774  jobs_done=0
Taxonomy: not found (skills stored as-is)
Utilities ready.


In [5]:
# Cell 4 — Async vLLM client (text-only)
async def call_vllm(session: aiohttp.ClientSession, system_msg: str, user_msg: str, max_tokens: int) -> str:
    payload = {
        "model": VLLM_MODEL, "max_tokens": max_tokens, "temperature": 0.0,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user",   "content": user_msg},
        ],
    }
    last_err = None
    for attempt in range(MAX_RETRIES + 1):
        try:
            async with session.post(
                f"{VLLM_BASE_URL}/chat/completions", json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT_S)
            ) as resp:
                if resp.status == 200:
                    data = await resp.json()
                    return data["choices"][0]["message"]["content"].strip()
                last_err = RuntimeError(f"HTTP {resp.status}: {(await resp.text())[:200]}")
        except asyncio.TimeoutError:
            last_err = RuntimeError(f"Timeout after {REQUEST_TIMEOUT_S}s")
        except Exception as e:
            last_err = e
        if attempt < MAX_RETRIES:
            await asyncio.sleep(RETRY_BACKOFF_S * (2 ** attempt))
    raise RuntimeError(f"vLLM failed after {MAX_RETRIES+1} attempts: {last_err}")

print("vLLM client defined.")

vLLM client defined.


In [6]:
# Cell 5 — Candidate extraction: prompts + record builder

CANDIDATE_SYSTEM = """You are a precise resume data extraction engine.
The input is OCR-extracted markdown from a scanned resume.
Return ONLY valid JSON. No markdown fences. No explanation. No preamble.
Use null for missing fields. Use [] for missing lists.
Extract only what is explicitly present in the text."""

def build_candidate_prompt(text: str) -> str:
    return f"""Extract structured data from this resume.

RESUME TEXT:
{text[:RESUME_TEXT_TRUNCATE]}

Return JSON with EXACTLY these fields:
{{
  "name": "full name or null",
  "email": "email or null",
  "phone": "phone or null",
  "location": {{"city": "or null", "country": "or null"}},
  "current_title": "most recent job title or null",
  "summary": "professional summary or null",
  "total_years_experience": <integer or null>,
  "seniority": "fresher|junior|mid|senior|lead|executive or null",
  "skills": {{
    "technical": ["technical skills"],
    "soft": ["soft skills"],
    "tools": ["tools and software"],
    "languages": ["programming languages"]
  }},
  "work_experience": [
    {{"company": "name", "title": "job title", "start_date": "YYYY-MM or null",
      "end_date": "YYYY-MM or Present", "duration_months": <integer or null>,
      "description": "one sentence summary"}}
  ],
  "education": [
    {{"institution": "name", "degree": "Bachelor's etc",
      "field": "field or null", "graduation_year": <integer or null>}}
  ],
  "certifications": ["list"],
  "languages_spoken": ["spoken languages"]
}}"""

def infer_seniority(years, title):
    t = (title or "").lower()
    for kw in ["executive","vp ","vice president","director","chief","cto","ceo","coo","cfo"]:
        if kw in t: return "executive"
    for kw in ["principal","staff ","lead ","head of","manager","architect"]:
        if kw in t: return "lead"
    for kw in ["senior","sr.","sr ","sr-"]: 
        if kw in t: return "senior"
    for kw in ["junior","jr.","associate","entry level","entry-level"]:
        if kw in t: return "junior"
    for kw in ["intern","trainee","fresher","graduate trainee"]:
        if kw in t: return "fresher"
    if years is None: return "unknown"
    if years == 0: return "fresher"
    if years <= 2: return "junior"
    if years <= 5: return "mid"
    if years <= 9: return "senior"
    return "lead"

def compute_quality_score(profile, text):
    s = 0
    if profile.get("name"): s += 5
    if profile.get("email") or profile.get("phone"): s += 5
    if profile.get("current_title"): s += 10
    tech = (profile.get("skills") or {}).get("technical", [])
    s += 20 if len(tech) >= 3 else (10 if tech else 0)
    exp = profile.get("work_experience") or []
    s += 20 if len(exp) >= 2 else (12 if len(exp) == 1 else 0)
    if profile.get("education"): s += 10
    if profile.get("summary"): s += 10
    ar = sum(1 for c in text if c.isalpha()) / max(len(text), 1)
    s += int(min(ar * 1.5, 1.0) * 20)
    return min(s, 100)

def build_embedding_text_candidate(profile, content):
    """Text that NB03 will encode into a vector. Mirrors job embedding structure."""
    parts = []
    if profile.get("current_title"): parts.append(profile["current_title"])
    sd = profile.get("skills") or {}
    all_skills = sd.get("technical",[]) + sd.get("tools",[]) + sd.get("languages",[])
    if all_skills: parts.append("Skills: " + ", ".join(all_skills[:30]))
    if profile.get("summary"): parts.append(profile["summary"][:300])
    for exp in (profile.get("work_experience") or [])[:3]:
        line = f"{exp.get('title','')} at {exp.get('company','')}: {exp.get('description','')}"
        parts.append(line[:200])
    return "\n".join(parts)[:1500]

def build_candidate_record(raw_record, profile):
    meta    = raw_record.get("metadata", {})
    content = raw_record.get("content", "")

    sd = profile.get("skills") or {}
    for key in ("technical","tools","soft","languages"):
        if sd.get(key): sd[key] = normalize_skills(sd[key])

    tech = sd.get("technical") or []
    tools = sd.get("tools") or []
    soft = sd.get("soft") or []

    skills_flat = list(dict.fromkeys(tech + tools + soft))

    years    = profile.get("total_years_experience")
    title    = profile.get("current_title")
    seniority = profile.get("seniority") or infer_seniority(years, title)

    # 🔥 NEW: missing fields detection
    missing_fields = []
    if not profile.get("name"): missing_fields.append("name")
    if not profile.get("email"): missing_fields.append("email")
    if not profile.get("phone"): missing_fields.append("phone")
    if not title: missing_fields.append("current_title")
    if not profile.get("summary"): missing_fields.append("summary")
    if not years: missing_fields.append("total_years_experience")
    if not skills_flat: missing_fields.append("skills")
    if not profile.get("education"): missing_fields.append("education")
    if not profile.get("work_experience"): missing_fields.append("work_experience")

    quality_score = compute_quality_score(profile, content)

    return {
        "candidate_id"          : raw_record["candidate_id"],
        "source_file"           : meta.get("pdf_path"),
        "source_category"       : meta.get("category"),
        "indexed_at"            : datetime.now(timezone.utc).isoformat(),

        "name"                  : profile.get("name"),
        "email"                 : profile.get("email"),
        "phone"                 : profile.get("phone"),
        "location"              : profile.get("location") or {},

        "current_title"         : title,
        "seniority"             : seniority,
        "total_years_experience": years,
        "summary"               : profile.get("summary"),

        "skills"                : sd,
        "skills_flat"           : skills_flat,
        "work_experience"       : profile.get("work_experience") or [],
        "education"             : profile.get("education") or [],
        "certifications"        : normalize_skills(profile.get("certifications") or []),
        "languages_spoken"      : profile.get("languages_spoken") or [],

        "raw_text"              : content,
        "page_count"            : meta.get("page_count", 1),

        "profile_quality_score" : quality_score,

        "is_poor_quality"       : quality_score < 50,
        "missing_fields"        : missing_fields,

        "extraction_method"     : "vllm-ocr+vllm-text-extraction",
        "embedding_text"        : build_embedding_text_candidate(profile, content),
        "description_vector"    : None,
    }

In [7]:
# Cell 6 — Run candidate extraction
import nest_asyncio
from tqdm.auto import tqdm
nest_asyncio.apply()

enriched_records = []
with open(ENRICHED_JSONL, "rb") as f:
    for line in f:
        if line.strip(): enriched_records.append(orjson.loads(line))
print(f"Loaded {len(enriched_records)} enriched resumes from NB01")

done_cand_ids   = set(progress.get("candidates_done", []))
remaining_cands = [r for r in enriched_records if r["candidate_id"] not in done_cand_ids]
print(f"Done={len(done_cand_ids)}  Remaining={len(remaining_cands)}")

async def _extract_one_candidate(raw_record, session, out_fh, skip_fh, lock):
    cid     = raw_record["candidate_id"]
    content = raw_record.get("content", "")

    #keep ONLY this skip
    if len(content.strip()) < MIN_RESUME_CHARS:
        async with lock:
            skip_fh.write(orjson.dumps({"candidate_id": cid, "reason": "text_too_short", "chars": len(content)}) + b"\n")
        return "skipped"

    raw_llm = await call_vllm(session, CANDIDATE_SYSTEM, build_candidate_prompt(content), CANDIDATE_MAX_TOKENS)

    profile = parse_llm_json(raw_llm)

    #fallback instead of skip
    if not profile:
        profile = {
            "name": None,
            "email": None,
            "phone": None,
            "location": {},
            "current_title": None,
            "summary": None,
            "total_years_experience": None,
            "seniority": None,
            "skills": {
                "technical": [],
                "soft": [],
                "tools": [],
                "languages": []
            },
            "work_experience": [],
            "education": [],
            "certifications": [],
            "languages_spoken": []
        }

    record = build_candidate_record(raw_record, profile)

    async with lock:
        out_fh.write(orjson.dumps(record) + b"\n")
        done_cand_ids.add(cid)
        progress["candidates_done"].append(cid)

    return "written"

async def run_candidate_extraction():
    start = time.time(); counts = {"written":0,"skipped":0,"error":0}; processed = 0
    lock = asyncio.Lock(); sem = asyncio.Semaphore(CONCURRENT_REQUESTS)
    out_fh  = open(CANDIDATES_JSONL,    "ab")
    skip_fh = open(SKIPPED_CANDS_JSONL, "ab")
    connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS+4)
    async with aiohttp.ClientSession(connector=connector) as session:
        pbar = tqdm(total=len(remaining_cands), desc="Candidates", unit="resume")
        async def process_one(rec):
            nonlocal processed
            async with sem:
                try:
                    outcome = await _extract_one_candidate(rec, session, out_fh, skip_fh, lock)
                except Exception as e:
                    outcome = "error"
                    async with lock: print(f"\n  ERROR {rec['candidate_id']}: {e}")
                finally:
                    async with lock:
                        counts[outcome] = counts.get(outcome,0) + 1; processed += 1
                        elapsed = time.time()-start; rate = max(processed,1)/max(elapsed,1)
                        eta = (len(remaining_cands)-processed)/max(rate,0.001)/60
                        pbar.update(1); pbar.set_postfix({"written":counts["written"],"skip":counts["skipped"],"err":counts["error"],"ETA_min":f"{eta:.0f}"})
                        if processed % CHECKPOINT_EVERY == 0:
                            out_fh.flush(); skip_fh.flush(); save_progress()
        await asyncio.gather(*[process_one(r) for r in remaining_cands])
        pbar.close()
    out_fh.flush(); out_fh.close(); skip_fh.flush(); skip_fh.close(); save_progress()
    elapsed = time.time()-start
    print(f"\nCandidate extraction complete.")
    print(f"  Written={counts['written']}  Skipped={counts['skipped']}  Errors={counts['error']}")
    print(f"  Time={elapsed/60:.1f}min  Avg={(elapsed/max(counts['written'],1)):.1f}s/resume")
    print(f"  Output: {CANDIDATES_JSONL}")

asyncio.run(run_candidate_extraction())

Loaded 2786 enriched resumes from NB01
Done=2760  Remaining=12


Candidates:  33%|███▎      | 4/12 [00:17<00:34,  4.32s/resume, written=0, skip=3, err=1, ETA_min=1]  


  ERROR 0228: can only concatenate list (not "NoneType") to list


Candidates:  42%|████▏     | 5/12 [00:21<00:29,  4.18s/resume, written=0, skip=3, err=2, ETA_min=0]


  ERROR 1559: can only concatenate list (not "NoneType") to list


...


  ERROR 2010: can only concatenate list (not "NoneType") to list


Candidates:  92%|█████████▏| 11/12 [00:54<00:05,  5.65s/resume, written=0, skip=3, err=8, ETA_min=0]


  ERROR 2020: can only concatenate list (not "NoneType") to list


Candidates: 100%|██████████| 12/12 [01:01<00:00,  5.13s/resume, written=0, skip=3, err=9, ETA_min=0]


  ERROR 2595: can only concatenate list (not "NoneType") to list

Candidate extraction complete.
  Written=0  Skipped=3  Errors=9
  Time=1.0min  Avg=61.6s/resume
  Output: ../data/processed/candidates.jsonl


In [8]:
# Cell 7 — Load + clean job descriptions CSV
import pandas as pd

def clean_job_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"equal\s+opportunity\s+employer", "", text, flags=re.IGNORECASE)
    text = re.sub(r"all\s+qualified\s+applicants.{0,120}consideration", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\{['\"][A-Za-z]+['\"].*?\}", "", text, flags=re.DOTALL)
    text = re.sub(r"[^\x09\x0A\x0D\x20-\x7E]", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def parse_skills_csv(raw) -> list:
    s = safe_str(raw)
    if not s: return []
    return [p.strip().strip("•·-").strip() for p in re.split(r"[\n,;/|]+", s) if len(p.strip()) > 1]

def parse_salary(raw) -> dict:
    result = {"min": None, "max": None, "currency": "USD", "raw": None}
    s = safe_str(raw)
    if not s: return result
    result["raw"] = s
    nums = [float(n) * (1000 if "k" in s.lower() else 1) for n in re.findall(r"[\d]+(?:\.\d+)?", s.replace(",","")) if n]
    if len(nums) >= 2: result["min"],result["max"] = int(nums[0]),int(nums[1])
    elif len(nums) == 1: result["min"] = result["max"] = int(nums[0])
    return result

def parse_experience(raw) -> dict:
    result = {"min_years": None, "max_years": None, "raw": None}
    s = safe_str(raw)
    if not s: return result
    result["raw"] = s
    nums = re.findall(r"\d+", s)
    if len(nums) >= 2: result["min_years"],result["max_years"] = int(nums[0]),int(nums[1])
    elif len(nums) == 1: result["min_years"] = result["max_years"] = int(nums[0])
    return result

def parse_posted_date(raw) -> str | None:
    s = safe_str(raw)
    if not s: return None
    for fmt in ("%d-%m-%Y","%Y-%m-%d","%m/%d/%Y","%d/%m/%Y","%Y/%m/%d"):
        try: return datetime.strptime(s, fmt).isoformat() + "Z"
        except: pass
    return None

print(f"Loading {JOBS_CSV} ...")
df = pd.read_csv(JOBS_CSV, low_memory=False, encoding="utf-8", on_bad_lines="skip")
df.columns = df.columns.str.strip()
print(f"Loaded {len(df):,} rows  |  Columns: {list(df.columns)}")

for col in ["Job Title","Role","Job Description","skills","Responsibilities","Company"]:
    if col in df.columns: df[col] = df[col].fillna("").astype(str)

# Drop rows empty in both title and description
mask = ~((df.get("Job Title", pd.Series(dtype=str)).str.strip() == "") &
         (df.get("Job Description", pd.Series(dtype=str)).str.strip() == ""))
df = df[mask].copy()
print(f"After empty drop : {len(df):,}")

dedup_cols = [c for c in ["Job Title","Company","location"] if c in df.columns]
before = len(df)
df = df.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
print(f"After dedup      : {len(df):,} (removed {before-len(df):,})")

# Apply row cap from config — done after dedup so IDs are stable
if MAX_JOBS is not None and len(df) > MAX_JOBS:
    df = df.iloc[:MAX_JOBS].reset_index(drop=True)
    print(f"After MAX_JOBS cap : {len(df):,}")

print(f"\nFinal job count to process : {len(df):,}")
print(f"\nTop 10 titles:")
print(df["Job Title"].value_counts().head(10).to_string())

Loading ../data/raw/job_descriptions.csv ...
Loaded 1,615,940 rows  |  Columns: ['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location', 'Country', 'latitude', 'longitude', 'Work Type', 'Company Size', 'Job Posting Date', 'Preference', 'Contact Person', 'Contact', 'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits', 'skills', 'Responsibilities', 'Company', 'Company Profile']
After empty drop : 1,615,940
After dedup      : 1,558,906 (removed 57,034)
After MAX_JOBS cap : 5,000

Final job count to process : 5,000

Top 10 titles:
Job Title
UX/UI Designer                  154
Software Engineer                87
Digital Marketing Specialist     85
Network Engineer                 69
Procurement Manager              68
Customer Support Specialist      67
Software Tester                  63
HR Coordinator                   57
Executive Assistant              57
Purchasing Agent                 57


In [9]:
# Cell 8 — Job extraction: prompts + record builder

JOB_SYSTEM = """You are a precise job listing data extraction engine.
Return ONLY valid JSON. No markdown fences. No explanation. No preamble.
Use null for missing fields. Use [] for missing lists.
Infer domain_tags purely from the job content — never use folder names or category labels."""

def build_job_prompt(title, role, desc, resp, skills_raw, experience):
    return f"""Extract structured data from this job listing.

JOB TITLE: {title}
ROLE: {role}
EXPERIENCE REQUIRED: {experience}
SKILLS LISTED: {skills_raw[:400]}
DESCRIPTION:
{desc[:JOB_TEXT_TRUNCATE]}
RESPONSIBILITIES:
{resp[:600]}

Return JSON with EXACTLY these fields:
{{
  "normalized_title"         : "concise standardized job title (e.g. 'Data Engineer', 'Frontend Developer')",
  "domain_tags"              : ["2-4 domain keywords inferred from content, e.g. 'data science', 'frontend', 'devops'"],
  "required_skills"          : ["skills that are clearly required"],
  "nice_to_have_skills"      : ["skills mentioned as preferred or bonus"],
  "tech_stack"               : ["specific technologies, frameworks, tools mentioned"],
  "seniority"                : "fresher|junior|mid|senior|lead|executive or null",
  "responsibilities_summary" : "2-3 sentence summary of what this person will do",
  "industry"                 : "industry sector or null"
}}"""

def build_embedding_text_job(row, llm_data):
    """Symmetric to candidate embedding — NB03 encodes this."""
    parts = []
    t = llm_data.get("normalized_title") or safe_str(row.get("Job Title"))
    if t: parts.append(t)
    skills = (llm_data.get("required_skills") or []) + (llm_data.get("tech_stack") or [])
    if skills: parts.append("Skills: " + ", ".join(skills[:30]))
    if llm_data.get("responsibilities_summary"): parts.append(llm_data["responsibilities_summary"][:300])
    desc = clean_job_text(safe_str(row.get("Job Description","")))
    if desc: parts.append(desc[:400])
    return "\n".join(parts)[:1500]

def build_job_record(row, llm_data, job_id):
    title      = safe_str(row.get("Job Title"))
    role       = safe_str(row.get("Role"))
    clean_desc = clean_job_text(safe_str(row.get("Job Description","")))
    clean_resp = clean_job_text(safe_str(row.get("Responsibilities","")))
    skills_raw = safe_str(row.get("skills",""))
    csv_skills     = parse_skills_csv(skills_raw)
    req_skills     = normalize_skills(llm_data.get("required_skills") or csv_skills)
    nice_to_have   = normalize_skills(llm_data.get("nice_to_have_skills") or [])
    tech_stack     = normalize_skills(llm_data.get("tech_stack") or [])
    # Domain tags from LLM only — no folder labels
    domain_tags = llm_data.get("domain_tags") or []
    if not domain_tags:
        norm = (llm_data.get("normalized_title") or title).lower()
        domain_tags = [w for w in re.split(r"\W+", norm) if len(w) > 3][:3]
    skills_flat = list(dict.fromkeys(req_skills + tech_stack))
    # Parse Company Profile JSON (stored as Python dict literal in CSV)
    company_profile = {}
    raw_cp = safe_str(row.get("Company Profile",""))
    if raw_cp:
        try: company_profile = json.loads(raw_cp.replace("'", '"'))
        except: pass
    return {
        "job_id"                  : job_id,
        "indexed_at"              : datetime.now(timezone.utc).isoformat(),
        "is_active"               : True,
        "title"                   : title,
        "normalized_title"        : llm_data.get("normalized_title") or title,
        "role"                    : role,
        "company"                 : safe_str(row.get("Company")) or None,
        "company_profile"         : company_profile,
        "industry"                : llm_data.get("industry") or None,
        "work_type"               : safe_str(row.get("Work Type")) or None,
        "company_size"            : row.get("Company Size"),
        "location"                : {
            "city"     : safe_str(row.get("location")) or None,
            "country"  : safe_str(row.get("Country"))  or None,
            "latitude" : row.get("latitude"),
            "longitude": row.get("longitude"),
        },
        "salary"                  : parse_salary(row.get("Salary Range")),
        "experience_required"     : parse_experience(row.get("Experience")),
        "qualifications"          : safe_str(row.get("Qualifications")) or None,
        "preference"              : safe_str(row.get("Preference")) or None,
        # domain_tags inferred by LLM from content only
        "required_skills"         : req_skills,
        "nice_to_have_skills"     : nice_to_have,
        "tech_stack"              : tech_stack,
        "skills_flat"             : skills_flat,
        "domain_tags"             : domain_tags,
        "seniority"               : llm_data.get("seniority"),
        "responsibilities_summary": llm_data.get("responsibilities_summary"),
        "description"             : clean_desc,
        "responsibilities"        : clean_resp,
        "benefits"                : safe_str(row.get("Benefits")) or None,
        "raw_text"                : f"{title}\n{role}\n{clean_desc}\n{clean_resp}",
        "posted_at"               : parse_posted_date(row.get("Job Posting Date")),
        "job_portal"              : safe_str(row.get("Job Portal")) or None,
        "contact_person"          : safe_str(row.get("Contact Person")) or None,
        # NB03 fills description_vector from this text
        "embedding_text"          : build_embedding_text_job(row, llm_data),
        "description_vector"      : None,
    }

print("Job extraction functions defined.")

Job extraction functions defined.


In [10]:
# Cell 9 — Run job extraction
# job_manifest = []
# for _, row in df.iterrows():
#     job_id = "job_" + hashlib.md5(
#         f"{safe_str(row.get('Job Title'))}|{safe_str(row.get('Company'))}|{safe_str(row.get('location'))}".encode()
#     ).hexdigest()[:12]
#     job_manifest.append((job_id, row))

job_manifest = []
for idx, (_, row) in enumerate(df.iterrows()):
    job_id = f"job_{str(idx + 1).zfill(4)}"   # job_0001, job_0002, ... job_5000
    job_manifest.append((job_id, row))

done_job_ids   = set(progress.get("jobs_done", []))
remaining_jobs = [(jid, row) for jid, row in job_manifest if jid not in done_job_ids]
print(f"Total={len(job_manifest):,}  Done={len(done_job_ids):,}  Remaining={len(remaining_jobs):,}")

async def _extract_one_job(job_id, row, session, out_fh, skip_fh, lock):
    title      = safe_str(row.get("Job Title",""))
    role       = safe_str(row.get("Role",""))
    clean_desc = clean_job_text(safe_str(row.get("Job Description","")))
    clean_resp = clean_job_text(safe_str(row.get("Responsibilities","")))
    skills_raw = safe_str(row.get("skills",""))
    experience = safe_str(row.get("Experience",""))
    if len(clean_desc) < MIN_JOB_DESC_CHARS:
        async with lock:
            skip_fh.write(orjson.dumps({"job_id":job_id,"reason":"desc_too_short","chars":len(clean_desc)})+b"\n")
        return "skipped"
    raw_llm  = await call_vllm(session, JOB_SYSTEM,
                               build_job_prompt(title,role,clean_desc,clean_resp,skills_raw,experience),
                               JOB_MAX_TOKENS)
    llm_data = parse_llm_json(raw_llm) or {}  # CSV fallback if LLM fails
    record   = build_job_record(row, llm_data, job_id)
    if len(record["skills_flat"]) < MIN_SKILLS_COUNT:
        async with lock:
            skip_fh.write(orjson.dumps({"job_id":job_id,"reason":"no_skills"})+b"\n")
        return "skipped"
    async with lock:
        out_fh.write(orjson.dumps(record)+b"\n")
        done_job_ids.add(job_id)
        progress["jobs_done"].append(job_id)
    return "written"

async def run_job_extraction():
    start = time.time(); counts = {"written":0,"skipped":0,"error":0}; processed = 0
    lock = asyncio.Lock(); sem = asyncio.Semaphore(CONCURRENT_REQUESTS)
    out_fh  = open(JOBS_JSONL,         "ab")
    skip_fh = open(SKIPPED_JOBS_JSONL, "ab")
    connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS+4)
    async with aiohttp.ClientSession(connector=connector) as session:
        pbar = tqdm(total=len(remaining_jobs), desc="Jobs", unit="job")
        async def process_one(job_id, row):
            nonlocal processed
            async with sem:
                try:
                    outcome = await _extract_one_job(job_id, row, session, out_fh, skip_fh, lock)
                except Exception as e:
                    outcome = "error"
                    async with lock: print(f"\n  ERROR {job_id}: {e}")
                finally:
                    async with lock:
                        counts[outcome] = counts.get(outcome,0)+1; processed += 1
                        elapsed = time.time()-start; rate = max(processed,1)/max(elapsed,1)
                        eta = (len(remaining_jobs)-processed)/max(rate,0.001)/60
                        pbar.update(1)
                        pbar.set_postfix({"written":counts["written"],"skip":counts["skipped"],"ETA_min":f"{eta:.0f}"})
                        if processed % CHECKPOINT_EVERY == 0:
                            out_fh.flush(); skip_fh.flush(); save_progress()
        await asyncio.gather(*[process_one(jid, row) for jid, row in remaining_jobs])
        pbar.close()
    out_fh.flush(); out_fh.close(); skip_fh.flush(); skip_fh.close(); save_progress()
    elapsed = time.time()-start
    print(f"\nJob extraction complete.")
    print(f"  Written={counts['written']:,}  Skipped={counts['skipped']:,}  Errors={counts['error']:,}")
    print(f"  Time={elapsed/60:.1f}min  Avg={(elapsed/max(counts['written'],1)):.2f}s/job")
    print(f"  Output: {JOBS_JSONL}")

asyncio.run(run_job_extraction())

Total=5,000  Done=0  Remaining=5,000


Jobs: 100%|██████████| 5000/5000 [1:48:14<00:00,  1.30s/job, written=5000, skip=0, ETA_min=0]   


Job extraction complete.
  Written=5,000  Skipped=0  Errors=0
  Time=108.2min  Avg=1.30s/job
  Output: ../data/processed/jobs.jsonl


In [11]:
# Cell 10 — Quality check: candidates
import random
from collections import Counter

candidates = []
with open(CANDIDATES_JSONL, "rb") as f:
    for line in f:
        if line.strip(): candidates.append(orjson.loads(line))

print(f"Total candidates : {len(candidates):,}")
scores = [c["profile_quality_score"] for c in candidates]
buckets = {"0-40":0,"41-60":0,"61-80":0,"81-100":0}
for s in scores:
    if s<=40: buckets["0-40"]+=1
    elif s<=60: buckets["41-60"]+=1
    elif s<=80: buckets["61-80"]+=1
    else: buckets["81-100"]+=1
print("\nQuality score distribution:")
for b,n in buckets.items():
    print(f"  {b:8s} {n:>5}  {'█'*(n*40//max(len(candidates),1))}")
print("\nSeniority distribution:")
for s,n in Counter(c["seniority"] for c in candidates).most_common():
    print(f"  {str(s):12s} {n:>5}")
print(f"\nEmbedding text ready : {sum(1 for c in candidates if c.get('embedding_text'))} / {len(candidates)}")
c = random.choice(candidates)
print(f"\nSample: {c['candidate_id']} | {c['name']} | {c['current_title']} | Q={c['profile_quality_score']}")
print(f"  Skills: {c['skills_flat'][:8]}")
print(f"  Embed : {(c.get('embedding_text') or '')[:120]}...")

Total candidates : 2,887

Quality score distribution:
  0-40       281  ███
  41-60      166  ██
  61-80     1036  ██████████████
  81-100    1404  ███████████████████

Seniority distribution:
  senior        1032
  lead           737
  mid            462
  unknown        293
  fresher        175
  executive      116
  junior          28
  Senior          23
  Lead            19
  intern           2

Embedding text ready : 2620 / 2887

Sample: 0072 | Robert Smith | Client Advocate/Advisor | Q=80
  Skills: []
  Embed : Client Advocate/Advisor
Self-motivated professional seeking a challenging position where I can utilize my strong persona...


In [12]:
# Cell 11 — Quality check: jobs
from collections import Counter
import random

jobs = []
with open(JOBS_JSONL, "rb") as f:
    for line in f:
        if line.strip(): jobs.append(orjson.loads(line))

print(f"Total jobs : {len(jobs):,}")
all_tags = [t for j in jobs for t in j.get("domain_tags",[])]
print(f"\nTop 15 domain tags (LLM-inferred from content only):")
for tag,n in Counter(all_tags).most_common(15):
    print(f"  {tag:40s} {n:>6}")
print(f"\nSeniority:")
for s,n in Counter(j.get("seniority") for j in jobs).most_common():
    print(f"  {str(s):12s} {n:>6}")
print(f"\nCoverage:")
print(f"  No skills_flat : {sum(1 for j in jobs if not j.get('skills_flat'))}")
print(f"  No domain_tags : {sum(1 for j in jobs if not j.get('domain_tags'))}")
print(f"  Embed ready    : {sum(1 for j in jobs if j.get('embedding_text'))} / {len(jobs)}")
j = random.choice(jobs)
print(f"\nSample: {j['job_id']} | {j['normalized_title']} | {j['company']} | {j['seniority']}")
print(f"  Required: {j['required_skills'][:5]}")
print(f"  Domains : {j['domain_tags']}")
print(f"  Embed   : {(j.get('embedding_text') or '')[:120]}...")

Total jobs : 5,000

Top 15 domain tags (LLM-inferred from content only):
  marketing                                   385
  finance                                     338
  sales                                       301
  supply chain                                266
  healthcare                                  251
  customer service                            229
  web development                             207
  architecture                                193
  software engineering                        179
  design                                      168
  digital marketing                           161
  information technology                      159
  data analysis                               145
  education                                   137
  nursing                                     136

Seniority:
  mid            4201
  fresher         417
  senior          250
  executive        81
  lead             51

Coverage:
  No skills_flat : 0
  No domain_tags : 0
  

In [ ]:
# Cell 12 — Summary
print("NB02 complete.")
print(f"  Candidates : {len(candidates):>6}  ->  {CANDIDATES_JSONL}")
print(f"  Jobs       : {len(jobs):>6}  ->  {JOBS_JSONL}")
print()
print("Both files contain:")
print("  embedding_text    : pre-computed text for vectorisation")
print("  description_vector: null (filled by NB03)")
print()
print("Next: Notebook 03 — Embedding Generation")
print("  Model  : BAAI/bge-large-en-v1.5  (768-dim, GPU batched)")
print("  Output : both files rewritten with description_vector populated")

NB02 complete.
  Candidates :   2887  ->  ../data/processed/candidates.jsonl
  Jobs       :   5000  ->  ../data/processed/jobs.jsonl

Both files contain:
  embedding_text    : pre-computed text for vectorisation
  description_vector: null (filled by NB03)

Next: Notebook 03 — Embedding Generation
  Model  : BAAI/bge-large-en-v1.5  (768-dim, GPU batched)
  Output : both files rewritten with description_vector populated


: 